# 04b — Social Media Charts (Altair + vl-convert)

Publication-ready PNG charts using the @unwelcomedata brand palette.
All charts export to twitter_landscape (1600×900px) with watermark.

**Production Charts:**
1. Top 10 Causes: Female vs Male (side-by-side, per 100k)
1b. Top 10 Causes: White vs Black (side-by-side, per 100k)
2. National Abortion Comparison (stacked male/female with gestation)
3-4. Abortion Impact by Race (White & Black)

In [ ]:
import sys
import os
from pathlib import Path

import pandas as pd
import duckdb
import yaml
import altair as alt

# Find project root
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT.parent / 'shared'))

from src.viz_social import save_social
from viz import PRESETS, SEX_COLORS, PALETTE

with open(PROJECT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

# Create outputs/social directory if needed
social_dir = PROJECT / 'outputs' / 'social'
social_dir.mkdir(parents=True, exist_ok=True)

# Connect to DuckDB
conn = duckdb.connect(str(PROJECT / 'data' / 'project.duckdb'))

# Get total abortions (national_total measure, 2024)
abort_total = conn.execute('''
  SELECT value
  FROM abortions
  WHERE measure = 'national_total' AND year = 2024
''').df()['value'].iloc[0]


# Display names for cleaner labels
DISPLAY_NAMES = {
    'Diseases of heart': 'Heart disease',
    'Malignant neoplasms': 'Cancer',
    'Chronic lower respiratory diseases': 'Respiratory disease',
    'Cerebrovascular diseases': 'Stroke',
    'Alzheimer disease': "Alzheimer's",
    'Diabetes mellitus': 'Diabetes',
    'Accidents (unintentional injuries)': 'Accidents',
    'Intentional self-harm (suicide)': 'Suicide',
    'Chronic liver disease and cirrhosis': 'Liver disease',
    'Nephritis, nephrotic syndrome and nephrosis': 'Kidney disease',
    'Influenza and pneumonia': 'Flu/Pneumonia',
    'Essential hypertension and hypertensive renal disease': 'Hypertension',
    'Assault (homicide)': 'Homicide',
    'Pregnancy, childbirth and the puerperium': 'Pregnancy/childbirth',
}

def short_name(cause: str) -> str:
    """Map verbose cause name to short display name."""
    return DISPLAY_NAMES.get(cause, cause)

print('✓ Environment loaded')
print(f'✓ Social charts will export to: {social_dir}')
print(f'✓ Total abortions 2024: {abort_total:,.0f}')

## Chart 1: Top 10 Causes of Death — Female vs Male (Side-by-Side)

Two horizontal bar panels placed side-by-side: Female top 10 on the left,
Male top 10 on the right. Each panel sorted independently by deaths descending.
This is the template for all side-by-side bar comparisons.

In [ ]:
# --- Chart 1 Data: Top 10 causes by sex (per-capita rates) ---

from chart_templates import side_by_side_bars

# Supplement display names for stripped versions
DISPLAY_NAMES['Intentional self-harm'] = 'Suicide'
DISPLAY_NAMES['Chronic liver disease and cirrhosis'] = 'Liver disease'
DISPLAY_NAMES['Assault'] = 'Homicide'
DISPLAY_NAMES['Parkinson disease'] = "Parkinson's"
DISPLAY_NAMES['Septicemia'] = 'Septicemia'

def clean_cause_raw(raw: str) -> str:
    """Strip # prefix and ICD code parenthetical from raw cause string."""
    name = raw.lstrip('#')
    if '(' in name:
        name = name[:name.index('(')].strip()
    return short_name(name)

def get_top10_rate_by_sex(sex):
    """Top 10 causes for a sex, ranked by deaths per 100k population."""
    df = conn.execute(f"""
      SELECT icd_10_113_cause_list as cause_raw,
             SUM(deaths) as deaths,
             SUM(population) as pop,
             ROUND(SUM(deaths) / SUM(population) * 100000, 1) as rate_per_100k
      FROM mortality_sex_age
      WHERE sex = '{sex}' AND icd_10_113_cause_list LIKE '#%'
      GROUP BY icd_10_113_cause_list
      ORDER BY rate_per_100k DESC
      LIMIT 10
    """).df()
    df['cause'] = df['cause_raw'].apply(clean_cause_raw)
    df = df.sort_values('rate_per_100k', ascending=False).reset_index(drop=True)
    df['total_label'] = df['rate_per_100k'].apply(lambda x: f'{x:.1f}')
    return df

def get_top10_rate_by_race(race):
    """Top 10 causes for a race (both sexes), ranked by deaths per 100k."""
    df = conn.execute(f"""
      SELECT icd_10_113_cause_list as cause_raw,
             SUM(deaths) as deaths,
             SUM(population) as pop,
             ROUND(SUM(deaths) / SUM(population) * 100000, 1) as rate_per_100k
      FROM mortality_race_sex
      WHERE single_race_6 = '{race}' AND icd_10_113_cause_list LIKE '#%'
      GROUP BY icd_10_113_cause_list
      ORDER BY rate_per_100k DESC
      LIMIT 10
    """).df()
    df['cause'] = df['cause_raw'].apply(clean_cause_raw)
    df = df.sort_values('rate_per_100k', ascending=False).reset_index(drop=True)
    df['total_label'] = df['rate_per_100k'].apply(lambda x: f'{x:.1f}')
    return df

def get_suicide_rank_female():
    """Find where suicide ranks among all female causes by rate.
    Uses deaths as tiebreaker for stable ranking."""
    all_f = conn.execute("""
      SELECT icd_10_113_cause_list as cause_raw,
             SUM(deaths) as deaths,
             ROUND(SUM(deaths) / SUM(population) * 100000, 1) as rate_per_100k
      FROM mortality_sex_age
      WHERE sex = 'Female' AND icd_10_113_cause_list LIKE '#%'
      GROUP BY icd_10_113_cause_list
      ORDER BY rate_per_100k DESC, deaths DESC
    """).df()
    all_f['cause'] = all_f['cause_raw'].apply(clean_cause_raw)
    all_f['rank'] = range(1, len(all_f) + 1)
    suicide_rows = all_f[all_f['cause'] == 'Suicide']
    if not suicide_rows.empty:
        return int(suicide_rows.iloc[0]['rank'])
    return None

# National per-capita data
df_female_top10 = get_top10_rate_by_sex('Female')
df_male_top10 = get_top10_rate_by_sex('Male')
suicide_female_rank = get_suicide_rank_female()

print('Female top 10 (per 100k):')
print(df_female_top10[['cause', 'rate_per_100k']].to_string(index=False))
print(f'\nMale top 10 (per 100k):')
print(df_male_top10[['cause', 'rate_per_100k']].to_string(index=False))
print(f'\nSuicide ranks #{suicide_female_rank} for women')

In [ ]:
# --- Build Chart 1: Side-by-side Female vs Male (per 100k) + detail bar ---

import altair as alt

x_max = max(df_female_top10['rate_per_100k'].max(), df_male_top10['rate_per_100k'].max()) * 1.15

# Male suicide age breakdown for detail bar
male_suicide_age = conn.execute("""
  SELECT 
    CASE 
      WHEN five_year_age_groups IN ('10-14 years', '15-19 years', '20-24 years') THEN '10-24'
      WHEN five_year_age_groups IN ('25-29 years', '30-34 years', '35-39 years', '40-44 years', '45-49 years') THEN '25-49'
      WHEN five_year_age_groups IN ('50-54 years', '55-59 years', '60-64 years', '65-69 years') THEN '50-69'
      ELSE '70+'
    END as age_band,
    SUM(deaths) as deaths
  FROM mortality_sex_age
  WHERE sex = 'Male' AND icd_10_113_cause_list LIKE '%self-harm%'
    AND five_year_age_groups NOT IN ('< 1 year', '1-4 years', '5-9 years', 'Not Stated')
  GROUP BY age_band
""").df()

total_ms = int(male_suicide_age['deaths'].sum())
age_order = ['10-24', '25-49', '50-69', '70+']
seg_ms = []
x_cursor = 0
for band in age_order:
    row = male_suicide_age[male_suicide_age['age_band'] == band]
    if row.empty: continue
    d = int(row['deaths'].iloc[0])
    pct = round(100 * d / total_ms)
    seg_ms.append({'segment': band, 'x_start': x_cursor, 'x_end': x_cursor + pct,
                   'pct_label': f'{band}: {pct}%', 'x_mid': x_cursor + pct/2})
    x_cursor += pct
# Force last segment to end at exactly 100
if seg_ms:
    seg_ms[-1]['x_end'] = 100
    seg_ms[-1]['x_mid'] = (seg_ms[-1]['x_start'] + 100) / 2
df_detail_ms = pd.DataFrame(seg_ms)

# --- Build panel specs ---
def build_panel_spec(df, color, panel_title, highlight_cat=None, hl_color=None, annotation=None):
    cause_order = df['cause'].tolist()
    if highlight_cat:
        df = df.copy()
        df['_bar_color'] = df['cause'].apply(lambda c: hl_color if c == highlight_cat else color)
    layers = []
    if highlight_cat:
        layers.append({
            'mark': {'type': 'bar', 'cornerRadiusEnd': 3},
            'encoding': {
                'y': {'field': 'cause', 'type': 'nominal', 'title': '', 'sort': cause_order,
                       'axis': {'labelFontSize': 13, 'labelFontWeight': 'bold', 'domain': False, 'ticks': False}},
                'x': {'field': 'rate_per_100k', 'type': 'quantitative', 'title': '', 'axis': None,
                       'scale': {'domain': [0, x_max]}},
                'color': {'field': '_bar_color', 'type': 'nominal', 'scale': None},
            },
        })
    else:
        layers.append({
            'mark': {'type': 'bar', 'cornerRadiusEnd': 3, 'color': color},
            'encoding': {
                'y': {'field': 'cause', 'type': 'nominal', 'title': '', 'sort': cause_order,
                       'axis': {'labelFontSize': 13, 'labelFontWeight': 'bold', 'domain': False, 'ticks': False}},
                'x': {'field': 'rate_per_100k', 'type': 'quantitative', 'title': '', 'axis': None,
                       'scale': {'domain': [0, x_max]}},
            },
        })
    layers.append({
        'mark': {'type': 'text', 'align': 'left', 'baseline': 'middle', 'dx': 6,
                 'fontSize': 13, 'fontWeight': 'bold', 'color': '#374151'},
        'encoding': {
            'y': {'field': 'cause', 'type': 'nominal', 'sort': cause_order},
            'x': {'field': 'rate_per_100k', 'type': 'quantitative'},
            'text': {'field': 'total_label', 'type': 'nominal'},
        },
    })
    if annotation and highlight_cat:
        hl_row = df[df['cause'] == highlight_cat]
        if not hl_row.empty:
            layers.append({
                'data': {'values': [{'cause': highlight_cat, 'x_pos': float(hl_row['rate_per_100k'].iloc[0]), 'label': annotation}]},
                'mark': {'type': 'text', 'align': 'left', 'baseline': 'middle', 'dx': 62,
                         'fontSize': 15, 'fontStyle': 'italic', 'color': '#6B7280'},
                'encoding': {
                    'y': {'field': 'cause', 'type': 'nominal', 'sort': cause_order},
                    'x': {'field': 'x_pos', 'type': 'quantitative'},
                    'text': {'field': 'label', 'type': 'nominal'},
                },
            })
    return {'data': {'values': df.to_dict('records')}, 'layer': layers,
            'width': 620, 'height': 420,
            'title': {'text': panel_title, 'anchor': 'start', 'offset': 8,
                      'fontSize': 16, 'fontWeight': 'bold', 'color': '#374151'}}

panel_female = build_panel_spec(df_female_top10, '#E9D8A6', 'Female')
panel_male = build_panel_spec(df_male_top10, '#005F73', 'Male',
                               highlight_cat='Suicide', hl_color='#94D2BD',
                               annotation=f'#{suicide_female_rank} for women')

# Detail bar: male suicide by age
age_colors = ['#94D2BD', '#0A9396', '#005F73', '#003049']
detail_bar_spec = {
    'data': {'values': df_detail_ms.to_dict('records')},
    'layer': [
        {'mark': {'type': 'bar', 'cornerRadiusEnd': 2, 'height': 30},
         'encoding': {
            'x': {'field': 'x_start', 'type': 'quantitative', 'axis': None, 'scale': {'domain': [0, 100]}},
            'x2': {'field': 'x_end'},
            'color': {'field': 'segment', 'type': 'nominal',
                      'scale': {'domain': age_order, 'range': age_colors}, 'legend': None},
         }},
        {'mark': {'type': 'text', 'fontSize': 13, 'fontWeight': 'bold', 'baseline': 'middle', 'color': '#ffffff'},
         'encoding': {'x': {'field': 'x_mid', 'type': 'quantitative'}, 'text': {'field': 'pct_label', 'type': 'nominal'}}},
    ],
    'width': 1280, 'height': 40,
    'title': {'text': 'Male suicide by age:', 'anchor': 'start', 'fontSize': 12,
              'fontWeight': 'normal', 'color': '#6B7280', 'offset': 2},
}

# Footer
rule_spec = {'data': {'values': [{}]}, 'mark': {'type': 'rule', 'color': '#D1D5DB', 'strokeWidth': 1},
             'encoding': {'y': {'value': 0}}, 'width': 1500, 'height': 2}
source_spec = {
    'data': {'values': [{'text': 'Source: CDC WONDER Mortality Data, 2024 · Crude rates per 100,000', 'x': 0}]},
    'mark': {'type': 'text', 'align': 'left', 'fontSize': 9, 'color': '#6B7280'},
    'encoding': {'x': {'field': 'x', 'type': 'quantitative', 'axis': None, 'scale': {'domain': [0, 1]}},
                 'text': {'field': 'text', 'type': 'nominal'}},
    'width': 1500, 'height': 18,
}

# Combine via vconcat
main_spec = {
    'hconcat': [panel_female, panel_male],
    'resolve': {'scale': {'x': 'shared'}},
    'spacing': 40,
    'title': {'text': 'Top 10 Causes of Death: Female vs Male (2024, per 100k)',
              'anchor': 'start', 'offset': 10},
}

chart1_spec = {
    '$schema': 'https://vega.github.io/schema/vega-lite/v5.json',
    'vconcat': [main_spec, detail_bar_spec, rule_spec, source_spec],
    'config': {
        'concat': {'spacing': 12},
        'axis': {'grid': False, 'domain': False, 'labelColor': '#374151'},
        'view': {'strokeWidth': 0},
    },
}

class _W:
    def __init__(self, s): self._spec = s
    def to_dict(self, *a, **k): return self._spec

chart1 = _W(chart1_spec)
print('Chart 1 built with detail bar')

In [ ]:
# Export Chart 1
save_social(chart1, cfg, '01_top10_causes_female_vs_male', preset='twitter_landscape')

from IPython.display import Image, display
display(Image(filename=str(social_dir / '01_top10_causes_female_vs_male.png')))

## Chart 1b: Top 10 Causes of Death — White vs Black (Side-by-Side)

Same side-by-side pattern but comparing races: White top 10 on the left,
Black top 10 on the right. Both use total deaths (male + female combined).

In [ ]:
# --- Chart 1b: White vs Black top 10 (per 100k) + detail bars ---

df_white_top10 = get_top10_rate_by_race('White')
df_black_top10 = get_top10_rate_by_race('Black or African American')

# Homicide rank for White
white_all_causes = conn.execute("""
  SELECT icd_10_113_cause_list as cause_raw, SUM(deaths) as deaths,
         ROUND(SUM(deaths) / SUM(population) * 100000, 1) as rate_per_100k
  FROM mortality_race_sex
  WHERE single_race_6 = 'White' AND icd_10_113_cause_list LIKE '#%'
  GROUP BY icd_10_113_cause_list ORDER BY rate_per_100k DESC, deaths DESC
""").df()
white_all_causes['cause'] = white_all_causes['cause_raw'].apply(clean_cause_raw)
white_all_causes['rank'] = range(1, len(white_all_causes) + 1)
homicide_white_rank = int(white_all_causes[white_all_causes['cause'] == 'Homicide'].iloc[0]['rank'])

# Suicide rank for Black
black_all_causes = conn.execute("""
  SELECT icd_10_113_cause_list as cause_raw, SUM(deaths) as deaths,
         ROUND(SUM(deaths) / SUM(population) * 100000, 1) as rate_per_100k
  FROM mortality_race_sex
  WHERE single_race_6 = 'Black or African American' AND icd_10_113_cause_list LIKE '#%'
  GROUP BY icd_10_113_cause_list ORDER BY rate_per_100k DESC, deaths DESC
""").df()
black_all_causes['cause'] = black_all_causes['cause_raw'].apply(clean_cause_raw)
black_all_causes['rank'] = range(1, len(black_all_causes) + 1)
suicide_black_rank = int(black_all_causes[black_all_causes['cause'] == 'Suicide'].iloc[0]['rank'])

# Detail bar 1: White suicide by age
white_suicide_age = conn.execute("""
  SELECT 
    CASE 
      WHEN five_year_age_groups IN ('10-14 years', '15-19 years', '20-24 years') THEN '10-24'
      WHEN five_year_age_groups IN ('25-29 years', '30-34 years', '35-39 years', '40-44 years', '45-49 years') THEN '25-49'
      WHEN five_year_age_groups IN ('50-54 years', '55-59 years', '60-64 years', '65-69 years') THEN '50-69'
      ELSE '70+'
    END as age_band,
    SUM(deaths) as deaths
  FROM mortality_race_age
  WHERE single_race_6 = 'White' AND icd_10_113_cause_list LIKE '%self-harm%'
    AND five_year_age_groups NOT IN ('< 1 year', '1-4 years', '5-9 years', 'Not Stated')
  GROUP BY age_band
""").df()
total_ws = int(white_suicide_age['deaths'].sum())
seg_ws = []
x = 0
for band in age_order:
    row = white_suicide_age[white_suicide_age['age_band'] == band]
    if row.empty: continue
    d = int(row['deaths'].iloc[0])
    pct = round(100 * d / total_ws)
    seg_ws.append({'segment': band, 'x_start': x, 'x_end': x + pct,
                   'pct_label': f'{band}: {pct}%', 'x_mid': x + pct/2})
    x += pct
# Force last segment to end at exactly 100
if seg_ws:
    seg_ws[-1]['x_end'] = 100
    seg_ws[-1]['x_mid'] = (seg_ws[-1]['x_start'] + 100) / 2
df_ws = pd.DataFrame(seg_ws)

# Detail bar 2: Black homicide by offender race
black_hom_off = conn.execute("""
  SELECT 
    CASE
      WHEN OffRace = 'Black' THEN 'Black'
      WHEN OffRace = 'White' THEN 'White'
      WHEN OffRace = 'Unknown' THEN 'Unknown'
      ELSE 'Other'
    END as offender,
    COUNT(*) as cases
  FROM shr_homicides_2024
  WHERE VicRace = 'Black'
  GROUP BY offender ORDER BY cases DESC
""").df()
total_bh = int(black_hom_off['cases'].sum())
off_order = ['Black', 'Unknown', 'White']
seg_bh = []
x = 0
for off in off_order:
    row = black_hom_off[black_hom_off['offender'] == off]
    if row.empty: continue
    cases = int(row['cases'].iloc[0])
    pct = round(100 * cases / total_bh)
    if pct < 1: continue
    seg_bh.append({'segment': off, 'x_start': x, 'x_end': x + pct,
                   'pct_label': f'{off}: {pct}%', 'x_mid': x + pct/2})
    x += pct
# Force last segment to end at exactly 100
if seg_bh:
    seg_bh[-1]['x_end'] = 100
    seg_bh[-1]['x_mid'] = (seg_bh[-1]['x_start'] + 100) / 2
df_bh = pd.DataFrame(seg_bh)

# Build panels
x_max_race = max(df_white_top10['rate_per_100k'].max(), df_black_top10['rate_per_100k'].max()) * 1.15

def build_race_panel(df, color, panel_title, highlight_cat=None, hl_color=None, annotation=None):
    cause_order = df['cause'].tolist()
    if highlight_cat:
        df = df.copy()
        df['_bar_color'] = df['cause'].apply(lambda c: hl_color if c == highlight_cat else color)
    layers = []
    if highlight_cat:
        layers.append({'mark': {'type': 'bar', 'cornerRadiusEnd': 3},
            'encoding': {'y': {'field': 'cause', 'type': 'nominal', 'title': '', 'sort': cause_order,
                               'axis': {'labelFontSize': 13, 'labelFontWeight': 'bold', 'domain': False, 'ticks': False}},
                         'x': {'field': 'rate_per_100k', 'type': 'quantitative', 'title': '', 'axis': None,
                               'scale': {'domain': [0, x_max_race]}},
                         'color': {'field': '_bar_color', 'type': 'nominal', 'scale': None}}})
    else:
        layers.append({'mark': {'type': 'bar', 'cornerRadiusEnd': 3, 'color': color},
            'encoding': {'y': {'field': 'cause', 'type': 'nominal', 'title': '', 'sort': cause_order,
                               'axis': {'labelFontSize': 13, 'labelFontWeight': 'bold', 'domain': False, 'ticks': False}},
                         'x': {'field': 'rate_per_100k', 'type': 'quantitative', 'title': '', 'axis': None,
                               'scale': {'domain': [0, x_max_race]}}}})
    layers.append({'mark': {'type': 'text', 'align': 'left', 'baseline': 'middle', 'dx': 6,
                            'fontSize': 13, 'fontWeight': 'bold', 'color': '#374151'},
                   'encoding': {'y': {'field': 'cause', 'type': 'nominal', 'sort': cause_order},
                                'x': {'field': 'rate_per_100k', 'type': 'quantitative'},
                                'text': {'field': 'total_label', 'type': 'nominal'}}})
    if annotation and highlight_cat:
        hl_row = df[df['cause'] == highlight_cat]
        if not hl_row.empty:
            layers.append({'data': {'values': [{'cause': highlight_cat, 'x_pos': float(hl_row['rate_per_100k'].iloc[0]), 'label': annotation}]},
                'mark': {'type': 'text', 'align': 'left', 'baseline': 'middle', 'dx': 62,
                         'fontSize': 15, 'fontStyle': 'italic', 'color': '#6B7280'},
                'encoding': {'y': {'field': 'cause', 'type': 'nominal', 'sort': cause_order},
                             'x': {'field': 'x_pos', 'type': 'quantitative'},
                             'text': {'field': 'label', 'type': 'nominal'}}})
    return {'data': {'values': df.to_dict('records')}, 'layer': layers,
            'width': 620, 'height': 420,
            'title': {'text': panel_title, 'anchor': 'start', 'offset': 8,
                      'fontSize': 16, 'fontWeight': 'bold', 'color': '#374151'}}

panel_white = build_race_panel(df_white_top10, '#CA6702', 'White',
                                highlight_cat='Suicide', hl_color='#BB3E03',
                                annotation=f'#{suicide_black_rank} for Black')
panel_black = build_race_panel(df_black_top10, '#EE9B00', 'Black',
                                highlight_cat='Homicide', hl_color='#BB3E03',
                                annotation=f'#{homicide_white_rank} for White')

# Detail bar specs
ws_colors = ['#EE9B00', '#CA6702', '#BB3E03', '#7C2D12']
detail_ws = {
    'data': {'values': df_ws.to_dict('records')},
    'layer': [
        {'mark': {'type': 'bar', 'cornerRadiusEnd': 2, 'height': 26},
         'encoding': {'x': {'field': 'x_start', 'type': 'quantitative', 'axis': None, 'scale': {'domain': [0, 100]}},
                      'x2': {'field': 'x_end'},
                      'color': {'field': 'segment', 'type': 'nominal',
                                'scale': {'domain': age_order, 'range': ws_colors}, 'legend': None}}},
        {'mark': {'type': 'text', 'fontSize': 12, 'fontWeight': 'bold', 'baseline': 'middle', 'color': '#ffffff'},
         'encoding': {'x': {'field': 'x_mid', 'type': 'quantitative'}, 'text': {'field': 'pct_label', 'type': 'nominal'}}},
    ],
    'width': 1280, 'height': 36,
    'title': {'text': 'White suicide by age:', 'anchor': 'start', 'fontSize': 12,
              'fontWeight': 'normal', 'color': '#6B7280', 'offset': 2},
}

bh_order = [s['segment'] for s in seg_bh]
bh_colors_map = {'Black': '#005F73', 'Unknown': '#94D2BD', 'White': '#CA6702'}
bh_colors = [bh_colors_map.get(s, '#6B7280') for s in bh_order]
detail_bh = {
    'data': {'values': df_bh.to_dict('records')},
    'layer': [
        {'mark': {'type': 'bar', 'cornerRadiusEnd': 2, 'height': 26},
         'encoding': {'x': {'field': 'x_start', 'type': 'quantitative', 'axis': None, 'scale': {'domain': [0, 100]}},
                      'x2': {'field': 'x_end'},
                      'color': {'field': 'segment', 'type': 'nominal',
                                'scale': {'domain': bh_order, 'range': bh_colors}, 'legend': None}}},
        {'mark': {'type': 'text', 'fontSize': 12, 'fontWeight': 'bold', 'baseline': 'middle'},
         'encoding': {'x': {'field': 'x_mid', 'type': 'quantitative'}, 'text': {'field': 'pct_label', 'type': 'nominal'},
                      'color': {'value': '#ffffff'}}},
    ],
    'width': 1280, 'height': 36,
    'title': {'text': 'Black homicide victims \u2014 offender race:', 'anchor': 'start', 'fontSize': 12,
              'fontWeight': 'normal', 'color': '#6B7280', 'offset': 2},
}

# Footer
rule_race = {'data': {'values': [{}]}, 'mark': {'type': 'rule', 'color': '#D1D5DB', 'strokeWidth': 1},
             'encoding': {'y': {'value': 0}}, 'width': 1500, 'height': 2}
source_race = {
    'data': {'values': [{'text': 'Source: CDC WONDER 2024 (mortality) \u00b7 FBI SHR 2024 (homicide offenders) \u00b7 Crude rates per 100,000', 'x': 0}]},
    'mark': {'type': 'text', 'align': 'left', 'fontSize': 9, 'color': '#6B7280'},
    'encoding': {'x': {'field': 'x', 'type': 'quantitative', 'axis': None, 'scale': {'domain': [0, 1]}},
                 'text': {'field': 'text', 'type': 'nominal'}},
    'width': 1500, 'height': 18,
}

chart1b_spec = {
    '$schema': 'https://vega.github.io/schema/vega-lite/v5.json',
    'vconcat': [{'hconcat': [panel_white, panel_black], 'resolve': {'scale': {'x': 'shared'}},
                 'spacing': 40, 'title': {'text': 'Top 10 Causes of Death: White vs Black (2024, per 100k)',
                                          'anchor': 'start', 'offset': 10}},
               detail_ws, detail_bh, rule_race, source_race],
    'config': {'concat': {'spacing': 8}, 'axis': {'grid': False, 'domain': False, 'labelColor': '#374151'},
              'view': {'strokeWidth': 0}},
}

chart1b = _W(chart1b_spec)

save_social(chart1b, cfg, '01b_top10_causes_white_vs_black', preset='twitter_landscape')

from IPython.display import Image, display
display(Image(filename=str(social_dir / '01b_top10_causes_white_vs_black.png')))

## Chart 2: National Abortion Comparison

"If abortion were a cause of death, it would rank as the #2-3 leading cause in the US"

In [ ]:
# Get top 5 causes nationally + abortion total
top_10_national = conn.execute('''
  SELECT 
    cause_raw,
    cause as cause_clean,
    deaths
  FROM mortality_national
  ORDER BY deaths DESC
  LIMIT 5
''').df()

# Apply display names
top_10_national['cause_display'] = top_10_national['cause_clean'].apply(short_name)

print(f'Top 5 causes loaded. #1: {top_10_national.iloc[0]["cause_display"]} ({top_10_national.iloc[0]["deaths"]:,.0f})')
print(f'Abortion total: {abort_total:,.0f} (would rank #{(top_10_national["deaths"] > abort_total).sum() + 1})')

In [ ]:
# Prepare Chart 2 data: stacked male/female bars with abortion inserted
#
# Structure: each cause gets two rows (Male segment, Female segment)
# Abortion gets a single row (solid bar, no sex split)
# We compute x_start and x_end for each segment so bar lengths are accurate.

# Get sex breakdown for all top 10 causes (join on cause_raw)
cause_raw_list = top_10_national['cause_raw'].tolist()
placeholders = ','.join([f"'{c}'" for c in cause_raw_list])
sex_by_cause = conn.execute(f'''
  SELECT 
    icd_10_113_cause_list as cause_raw,
    sex,
    SUM(deaths) as deaths
  FROM mortality_sex_age
  WHERE icd_10_113_cause_list IN ({placeholders})
  GROUP BY icd_10_113_cause_list, sex
''').df()

# Build the chart data: one entry per segment (Male/Female/Abortion)
rows = []
for _, row in top_10_national.iterrows():
    cause_raw = row['cause_raw']
    cause_display = row['cause_display']
    total_deaths = int(row['deaths'])
    
    sex_data = sex_by_cause[sex_by_cause['cause_raw'] == cause_raw]
    male_d = int(sex_data[sex_data['sex'] == 'Male']['deaths'].sum())
    female_d = int(sex_data[sex_data['sex'] == 'Female']['deaths'].sum())
    sex_total = male_d + female_d
    
    # Use sex_total for bar length (segments add up correctly)
    male_pct = round(100 * male_d / sex_total) if sex_total > 0 else 0
    female_pct = round(100 * female_d / sex_total) if sex_total > 0 else 0
    
    # Male segment: starts at 0, ends at male_d
    rows.append({
        'cause': cause_display,
        'segment': 'Male',
        'x_start': 0,
        'x_end': male_d,
        'seg_deaths': male_d,
        'total_deaths': total_deaths,
        'pct_label': f'{male_pct}%',
        'is_abortion': False,
    })
    # Female segment: starts at male_d, ends at sex_total
    rows.append({
        'cause': cause_display,
        'segment': 'Female',
        'x_start': male_d,
        'x_end': sex_total,
        'seg_deaths': female_d,
        'total_deaths': total_deaths,
        'pct_label': f'{female_pct}%',
        'is_abortion': False,
    })

# Insert Abortion row (single segment, no sex split)
rows.append({
    'cause': 'Abortion',
    'segment': 'Abortion',
    'x_start': 0,
    'x_end': int(abort_total),
    'seg_deaths': int(abort_total),
    'total_deaths': int(abort_total),
    'pct_label': '',  # No sex label for abortion
    'is_abortion': True,
})

df_chart2 = pd.DataFrame(rows)

# Compute midpoint for centering text inside each segment
df_chart2['x_mid'] = (df_chart2['x_start'] + df_chart2['x_end']) / 2

# Sort order: largest total at top (descending)
cause_totals = df_chart2.groupby('cause')['total_deaths'].first().reset_index()
cause_totals = cause_totals.sort_values('total_deaths', ascending=False)
cause_order = cause_totals['cause'].tolist()

# Build a separate df for total labels (one row per cause, at x_end of full bar)
df_totals = df_chart2.groupby('cause').agg(
    total_deaths=('total_deaths', 'first'),
    bar_end=('x_end', 'max')
).reset_index()
df_totals['total_label'] = df_totals['total_deaths'].apply(lambda x: f'{x:,}')

print(f'Chart 2 data prepared:')
print(f'  {len(cause_order)} causes (including Abortion)')
print(f'  Abortion: {abort_total:,.0f}')
print(f'  Heart disease: {top_10_national.iloc[0]["deaths"]:,.0f}')
print(f'\nSegment data sample:')
print(df_chart2[['cause', 'segment', 'x_start', 'x_end', 'pct_label']].head(6))
print(f'\nCause order (bottom to top): {cause_order}')

In [ ]:
# Build Chart 2: Stacked male/female bars with sex% inside, total count to right
# Uses x (start) and x2 (end) encoding for accurate proportional bar segments.

from viz import SEX_COLORS

# Color mapping: Male=light teal, Female=peach, Abortion=burnt caramel
segment_colors = {
    'Male': '#005F73',
    'Female': '#E9D8A6',
    'Abortion': '#AE2012',
}

# --- Layer 1: Stacked bars using x/x2 ---
bars = alt.Chart(df_chart2).mark_bar().encode(
    y=alt.Y('cause:N', title='', sort=cause_order,
            axis=alt.Axis(labelFontSize=14, labelFontWeight='bold', domain=False, ticks=False)),
    x=alt.X('x_start:Q', title='', axis=None, scale=alt.Scale(domain=[0, df_chart2['x_end'].max() * 1.12])),
    x2='x_end:Q',
    color=alt.Color('segment:N',
                    scale=alt.Scale(
                        domain=['Male', 'Female', 'Abortion'],
                        range=[segment_colors['Male'], segment_colors['Female'], segment_colors['Abortion']]
                    ),
                    legend=None),
).properties(
    width=1300,
    height=500,
    title={
        'text': 'What if abortion was counted as a cause of death?',
        'subtitle': 'Top 5 causes of death among all Americans, by sex (2024)',
        'anchor': 'start',
        'offset': 10,
    }
)

# --- Layer 2: Direct 'Male' / 'Female' labels inside Heart disease bar ---
heart_row = df_chart2[df_chart2['cause'] == 'Heart disease']
heart_male = heart_row[heart_row['segment'] == 'Male'].iloc[0]
heart_female = heart_row[heart_row['segment'] == 'Female'].iloc[0]

# Male label (light text on dark teal)
df_label_male = pd.DataFrame([{'cause': 'Heart disease', 'x_pos': heart_male['x_start'], 'label': 'Male'}])
text_label_male = alt.Chart(df_label_male).mark_text(
    align='left', baseline='top', dx=8, dy=-20,
    fontSize=13, fontWeight='bold', color='#E9D8A6'
).encode(y=alt.Y('cause:N', sort=cause_order), x=alt.X('x_pos:Q'), text='label:N')

# Female label (dark text on vanilla custard)
df_label_female = pd.DataFrame([{'cause': 'Heart disease', 'x_pos': heart_female['x_start'], 'label': 'Female'}])
text_label_female = alt.Chart(df_label_female).mark_text(
    align='left', baseline='top', dx=8, dy=-20,
    fontSize=13, fontWeight='bold', color='#003049'
).encode(y=alt.Y('cause:N', sort=cause_order), x=alt.X('x_pos:Q'), text='label:N')

# --- Layer 3: Sex percentage labels centered inside each segment ---
# Split into two layers for proper text color contrast
max_bar = df_chart2['x_end'].max()
min_segment_for_label = max_bar * 0.035
df_pct_labels = df_chart2[
    (df_chart2['is_abortion'] == False) &
    (df_chart2['seg_deaths'] >= min_segment_for_label)
].copy()

# Male pct labels (light text on dark teal)
df_pct_male = df_pct_labels[df_pct_labels['segment'] == 'Male']
text_pct_male = alt.Chart(df_pct_male).mark_text(
    align='center', baseline='middle', color='#E9D8A6', fontSize=13, fontWeight='bold'
).encode(y=alt.Y('cause:N', sort=cause_order), x=alt.X('x_mid:Q'), text='pct_label:N')

# Female pct labels (dark text on vanilla custard)
df_pct_female = df_pct_labels[df_pct_labels['segment'] == 'Female']
text_pct_female = alt.Chart(df_pct_female).mark_text(
    align='center', baseline='middle', color='#003049', fontSize=13, fontWeight='bold'
).encode(y=alt.Y('cause:N', sort=cause_order), x=alt.X('x_mid:Q'), text='pct_label:N')

# --- Layer 3: Total count label to the right of each bar ---
text_total = alt.Chart(df_totals).mark_text(
    align='left', baseline='middle', dx=8,
    fontSize=15, fontWeight='bold', color='#374151'
).encode(
    y=alt.Y('cause:N', sort=cause_order),
    x=alt.X('bar_end:Q'),
    text='total_label:N',
)

# --- Combine all layers ---
# First, add gestation age dividers within the abortion bar
# Get gestation data: <=9wk (78.6%), 10-13wk (14.2%), 14-20wk (6.1%), >=21wk (1.1%)
gest_data = conn.execute("""
    SELECT age_group, value FROM abortions
    WHERE measure = 'gestation_count' ORDER BY value DESC
""").df()
gest_pcts = conn.execute("""
    SELECT age_group, value FROM abortions
    WHERE measure = 'gestation_pct' ORDER BY value DESC
""").df()

# Build gestation segments (cumulative x positions, ordered earliest first)
gest_order = ['<=9_weeks', '10-13_weeks', '14-20_weeks', '>=21_weeks']
gest_labels = {'<=9_weeks': '≤9 wks', '10-13_weeks': '10-13 wks', '14-20_weeks': '14-20 wks', '>=21_weeks': '≥21 wks'}
mx = df_chart2['x_end'].max()
x_cursor = 0
gest_boundaries = []  # x positions for vertical rules
gest_label_rows = []  # for centered text labels
for g in gest_order:
    count = int(gest_data[gest_data['age_group'] == g]['value'].iloc[0])
    pct = float(gest_pcts[gest_pcts['age_group'] == g]['value'].iloc[0])
    x_start_g = x_cursor
    x_end_g = x_cursor + count
    x_mid_g = (x_start_g + x_end_g) / 2
    gest_label_rows.append({'cause': 'Abortion', 'x_mid': x_mid_g, 'label': gest_labels[g], 'pct': f'{pct:.0f}%'}) if count > mx * 0.10 else None
    x_cursor = x_end_g
    if x_cursor < int(abort_total):  # don't add rule after last segment
        gest_boundaries.append({'cause': 'Abortion', 'x_pos': x_cursor})

# Vertical rules at gestation boundaries (thick dark red lines)
df_gest_rules = pd.DataFrame(gest_boundaries)
# Add a tiny x_end for each rule (3px worth of the data scale)
px_width = (df_chart2['x_end'].max() * 1.12) / 1300 * 3  # ~3px in data units
df_gest_rules['x_end'] = df_gest_rules['x_pos'] + px_width
gest_rules = alt.Chart(df_gest_rules).mark_bar(
    color='#fff5e6',
).encode(
    x=alt.X('x_pos:Q'),
    x2='x_end:Q',
    y=alt.Y('cause:N', sort=cause_order),
) if len(df_gest_rules) > 0 else alt.Chart(pd.DataFrame()).mark_point()

# Gestation labels centered in each section
df_gest_labels = pd.DataFrame(gest_label_rows)
# Gestation name (above center)
gest_text_name = alt.Chart(df_gest_labels).mark_text(
    align='center', baseline='bottom', dy=-2, color='#E9D8A6', fontSize=11, fontWeight='bold'
).encode(
    y=alt.Y('cause:N', sort=cause_order),
    x=alt.X('x_mid:Q'),
    text='label:N',
)
# Gestation pct (below center)
gest_text_pct = alt.Chart(df_gest_labels).mark_text(
    align='center', baseline='top', dy=2, color='#E9D8A6', fontSize=11, fontWeight='bold'
).encode(
    y=alt.Y('cause:N', sort=cause_order),
    x=alt.X('x_mid:Q'),
    text='pct:N',
)

chart2 = (bars + text_label_male + text_label_female + text_pct_male + text_pct_female + text_total + gest_rules + gest_text_name + gest_text_pct).configure_axis(
    grid=False,
    domain=False,
    labelColor='#374151'
).configure_view(
    strokeWidth=0
)

print('Chart 1 built: stacked male/female bars + sex% inside + total to right')

In [ ]:
# Export Chart 2
from chart_templates import add_footer
chart2_final = add_footer(chart2, source='CDC WONDER 2024 (mortality) | Guttmacher Institute 2024 (abortion counts)')
save_social(chart2_final, cfg, '02_abortion_comparison_national', preset='twitter_landscape')
print('✓ Chart 2 exported')

from IPython.display import Image, display
display(Image(filename=str(social_dir / '02_abortion_comparison_national.png')))

## Charts 3 & 4: Top 5 Causes by Race (White & Black)

Same stacked male/female bar pattern, filtered by race.
Uses the  template with footer.

In [ ]:
# Charts 3 & 4: Top 5 causes by race (White, Black) — same format as Chart 2 with abortion
from chart_templates import add_footer

races = [
    ("White", "03_top5_causes_white"),
    ("Black or African American", "04_top5_causes_black"),
]
RACE_DISPLAY = {"White": "White", "Black or African American": "Black"}

# Race-specific abortion counts from DuckDB
# Source: Guttmacher Abortion Patient Survey 2021-2022 proportions applied to 2024 total
ABORT_BY_RACE = {
    "White": int(conn.execute("SELECT value FROM abortions WHERE measure='race_count' AND age_group='NH White'").df()['value'].iloc[0]),
    "Black or African American": int(conn.execute("SELECT value FROM abortions WHERE measure='race_count' AND age_group='Black'").df()['value'].iloc[0]),
}

for race_name, filename in races:
    race_display = RACE_DISPLAY[race_name]
    # Get top 5 causes for this race (only # prefixed = summary causes)
    top5_race = conn.execute(f"""
        SELECT icd_10_113_cause_list as cause_raw, SUM(deaths) as total_deaths
        FROM mortality_race_sex
        WHERE single_race_6 = '{race_name}'
          AND icd_10_113_cause_list LIKE '#%'
        GROUP BY icd_10_113_cause_list
        ORDER BY total_deaths DESC
        LIMIT 5
    """).df()

    # Get sex breakdown for those causes
    cause_list = top5_race["cause_raw"].tolist()
    ph = ",".join([f"'{c}'" for c in cause_list])
    sex_race = conn.execute(f"""
        SELECT icd_10_113_cause_list as cause_raw, sex, SUM(deaths) as deaths
        FROM mortality_race_sex
        WHERE single_race_6 = '{race_name}'
          AND icd_10_113_cause_list IN ({ph})
        GROUP BY icd_10_113_cause_list, sex
    """).df()

    # Build segment data (same structure as Chart 1)
    rows = []
    for _, row in top5_race.iterrows():
        cause_clean = row["cause_raw"].lstrip("#").split(" (")[0]
        cause_display = short_name(cause_clean)
        total_deaths = int(row["total_deaths"])
        sd = sex_race[sex_race["cause_raw"] == row["cause_raw"]]
        male_d = int(sd[sd["sex"] == "Male"]["deaths"].sum())
        female_d = int(sd[sd["sex"] == "Female"]["deaths"].sum())
        sex_total = male_d + female_d
        male_pct = round(100 * male_d / sex_total) if sex_total > 0 else 0
        female_pct = round(100 * female_d / sex_total) if sex_total > 0 else 0
        rows.append({"cause": cause_display, "segment": "Male", "x_start": 0, "x_end": male_d,
                     "seg_deaths": male_d, "total_deaths": total_deaths, "pct_label": f"{male_pct}%", "is_abortion": False})
        rows.append({"cause": cause_display, "segment": "Female", "x_start": male_d, "x_end": sex_total,
                     "seg_deaths": female_d, "total_deaths": total_deaths, "pct_label": f"{female_pct}%", "is_abortion": False})

    # Insert Abortion row
    rows.append({"cause": "Abortion", "segment": "Abortion", "x_start": 0, "x_end": ABORT_BY_RACE[race_name],
                 "seg_deaths": ABORT_BY_RACE[race_name], "total_deaths": ABORT_BY_RACE[race_name], "pct_label": "", "is_abortion": True})

    df_race = pd.DataFrame(rows)
    df_race["x_mid"] = (df_race["x_start"] + df_race["x_end"]) / 2

    # Sort: largest on top
    ct = df_race.groupby("cause")["total_deaths"].first().reset_index().sort_values("total_deaths", ascending=False)
    co = ct["cause"].tolist()
    df_t = df_race.groupby("cause").agg(total_deaths=("total_deaths", "first"), bar_end=("x_end", "max")).reset_index()
    df_t["total_label"] = df_t["total_deaths"].apply(lambda x: f"{x:,}")

    segment_colors = {"Male": "#005F73", "Female": "#E9D8A6", "Abortion": "#AE2012"}

    bars_r = alt.Chart(df_race).mark_bar().encode(
        y=alt.Y("cause:N", title="", sort=co, axis=alt.Axis(labelFontSize=14, labelFontWeight="bold", domain=False, ticks=False)),
        x=alt.X("x_start:Q", title="", axis=None, scale=alt.Scale(domain=[0, df_race["x_end"].max() * 1.12])),
        x2="x_end:Q",
        color=alt.Color("segment:N", scale=alt.Scale(domain=list(segment_colors.keys()), range=list(segment_colors.values())), legend=None),
    ).properties(width=1300, height=500,
        title={"text": "What if abortion was counted as a cause of death?",
               "subtitle": f"Top 5 causes of death among {race_display} Americans, by sex (2024)",
               "anchor": "start", "offset": 10})

    # Direct labels on first non-abortion cause
    first_cause = co[1] if co[0] == "Abortion" else co[0]
    fc_rows = df_race[df_race["cause"] == first_cause]
    fc_m = fc_rows[fc_rows["segment"] == "Male"].iloc[0]
    fc_f = fc_rows[fc_rows["segment"] == "Female"].iloc[0]

    tl_m = alt.Chart(pd.DataFrame([{"cause": first_cause, "x_pos": fc_m["x_start"], "label": "Male"}])).mark_text(
        align="left", baseline="top", dx=8, dy=-20, fontSize=13, fontWeight="bold", color="#E9D8A6"
    ).encode(y=alt.Y("cause:N", sort=co), x="x_pos:Q", text="label:N")

    tl_f = alt.Chart(pd.DataFrame([{"cause": first_cause, "x_pos": fc_f["x_start"], "label": "Female"}])).mark_text(
        align="left", baseline="top", dx=8, dy=-20, fontSize=13, fontWeight="bold", color="#003049"
    ).encode(y=alt.Y("cause:N", sort=co), x="x_pos:Q", text="label:N")

    # Pct labels (split for text color contrast)
    mx = df_race["x_end"].max()
    mn = mx * 0.035
    dpl = df_race[(~df_race["is_abortion"]) & (df_race["seg_deaths"] >= mn)]
    tp_m = alt.Chart(dpl[dpl["segment"] == "Male"]).mark_text(
        align="center", baseline="middle", color="#E9D8A6", fontSize=13, fontWeight="bold"
    ).encode(y=alt.Y("cause:N", sort=co), x="x_mid:Q", text="pct_label:N")
    tp_f = alt.Chart(dpl[dpl["segment"] == "Female"]).mark_text(
        align="center", baseline="middle", color="#003049", fontSize=13, fontWeight="bold"
    ).encode(y=alt.Y("cause:N", sort=co), x="x_mid:Q", text="pct_label:N")
    tt = alt.Chart(df_t).mark_text(
        align="left", baseline="middle", dx=8, fontSize=15, fontWeight="bold", color="#374151"
    ).encode(y=alt.Y("cause:N", sort=co), x="bar_end:Q", text="total_label:N")

    chart_r = (bars_r + tl_m + tl_f + tp_m + tp_f + tt).configure_axis(
        grid=False, domain=False, labelColor="#374151"
    ).configure_view(strokeWidth=0)

    chart_r_final = add_footer(chart_r, source="CDC WONDER 2024 (mortality) | Guttmacher Institute 2024 (abortion counts)")
    save_social(chart_r_final, cfg, filename, preset="twitter_landscape")
    print(f"✓ {filename} exported")

    from IPython.display import Image, display
    display(Image(filename=str(social_dir / f"{filename}.png")))


## Summary & Cleanup

In [ ]:
conn.close()

# Verify all exports
pngs = sorted(social_dir.glob('*.png'))
print('=== ALL CHARTS COMPLETE ===')
print(f'\n✓ Generated {len(pngs)} publication-ready charts:')
for png in pngs:
    size_kb = png.stat().st_size / 1024
    print(f'  • {png.name} ({size_kb:.0f} KB)')

print('\nAll charts are twitter_landscape (1600×900px) with @unwelcomedata watermark.')
print('Ready for social media posting!')